# 092 — Generación y edición de video

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** (a) n = 8 × 256 = **2 048** tokens. (b) 2 048² = **4 194 304** pares.
(c) espacial 8·256² = 524 288; temporal 256·8² = 16 384; total **540 672**.
(d) 4 194 304 / 540 672 ≈ **7.8×**.

**Ejercicio 2.** Al duplicar F: la atención completa escala con (F·hw)² → **×4**
(16 777 216); la espacial F·(hw)² es lineal en F → **×2** (1 048 576); la temporal
hw·F² es cuadrática en F → **×4** (65 536). Total factorizado 1 114 112; nueva razón
16 777 216 / 1 114 112 ≈ **15.1×**. Conclusión: el ahorro relativo de la factorización
CRECE con la longitud del clip, porque el término espacial (dominante) solo escala
linealmente con F.

**Ejercicio 3.** Frames independientes no comparten información: cada denoising
muestrea una moda distinta de apariencia → parpadeo de identidad, textura e
iluminación, aunque cada frame sea plausible. El componente mínimo es un mecanismo de
acoplamiento temporal — atención o convolución sobre el eje F (o al menos condicionar
cada frame en el anterior) — que es exactamente el término h·w·F² que la factorizada
añade sobre el costo espacial.

**Ejercicio 4.** El contrato JSON expone `kind` y `evidence`; solo esa evidencia
inspeccionable autoriza conclusiones.


In [ ]:
result = run_lab("generation", seed=92)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica de los ejercicios
def costos(F, hw):
    n = F * hw
    completa = n ** 2
    factorizada = F * hw**2 + hw * F**2
    return n, completa, factorizada

# Ejercicio 1: F=8, latentes 16x16
n, comp, fact = costos(8, 256)
print(f"F=8 : n={n}  completa={comp:,}  factorizada={fact:,}  razón={comp/fact:.1f}x")
assert (n, comp, fact) == (2_048, 4_194_304, 540_672)

# Ejercicio 2: F=16 — completa x4, espacial x2, temporal x4
n2, comp2, fact2 = costos(16, 256)
print(f"F=16: n={n2}  completa={comp2:,}  factorizada={fact2:,}  razón={comp2/fact2:.1f}x")
assert comp2 == 4 * comp
assert 16 * 256**2 == 2 * (8 * 256**2)   # espacial x2
assert 256 * 16**2 == 4 * (256 * 8**2)   # temporal x4

# Ejemplo del README: 16 frames de 32x32
n3, comp3, fact3 = costos(16, 1024)
print(f"README: n={n3}  completa={comp3:,}  factorizada={fact3:,}  razón={comp3/fact3:.1f}x")
assert (comp3, fact3) == (268_435_456, 17_039_360)


## Reflexión

1. En la atención factorizada, un token del frame 3 en la posición (0,0) y otro del
   frame 9 en la posición (31,31) nunca se atienden directamente: ¿por qué camino les
   llega la información y qué tipo de movimiento podría degradarse por ello?
2. Make-A-Video aprende apariencia de pares texto-imagen y movimiento de video SIN
   texto: ¿qué suposición sobre la factorización apariencia/movimiento hace posible
   ese truco y cuándo fallaría?
3. En video2video, ¿por qué ruidificar poco preserva el video original y ruidificar
   mucho lo destruye, y cómo elegirías el nivel para "cambiar el estilo sin cambiar
   el movimiento"?
